# Citation Index API — interactive client test (API 1.0.0-beta.1)

Run this notebook from the project root with the package installed:

```bash
pip install -e ".[dev]"
pip install ipykernel   # if needed
python -m ipykernel install --user --name citation-index-api-client
```

Every endpoint needs an API key (sent as `X-API-Key`). Set
`CITATION_INDEX_API_KEY` before starting Jupyter, or paste it into `API_KEY` below.

Select that kernel in Jupyter / VS Code, then run the cells top to bottom.

In [1]:
import os
from pathlib import Path

from citation_index_api_client import CitationIndexClient, __version__
from citation_index_api_client.exceptions import APIError, JobFailedError, JobTimeoutError

# --- configure ---
BASE_URL = "https://citation-index.graphia-ssh.eu"
API_KEY = os.environ.get("CITATION_INDEX_API_KEY")  # or paste a key here (don't commit it)
PDF_PATH = Path("paper.pdf")  # set to your PDF, or None to skip text extraction
POLL_INTERVAL = 2.0
JOB_TIMEOUT = 600.0

if not API_KEY:
    print("⚠ No API key — set CITATION_INDEX_API_KEY; every endpoint returns 401 without it")

client = CitationIndexClient(base_url=BASE_URL, api_key=API_KEY, timeout=120.0)
print(f"Client {__version__} → {client.base_url}")

Client 1.0.0b1 → https://citation-index.graphia-ssh.eu


## Health & API info

In [2]:
try:
    print("health:", client.health())
    print("root:", client.root())
except APIError as e:
    print(f"API error {e.status_code}: {e.detail}")

health: {'status': 'healthy', 'redis': 'ok', 'storage': 'ok', 'version': '1.0.0-beta.1'}
root: {'name': 'Citation Index API', 'version': '1.0.0-beta.1', 'docs': '/docs', 'health': '/health'}


## Extract text from a PDF

Set `PDF_PATH` in the first cell to a local `.pdf` file. This enqueues an async job and waits for the result.

In [3]:
PDF_PATH = Path("../test_data/reg_0035-2039_1989_num_102_485_2445.pdf")

In [4]:
if PDF_PATH is None or not PDF_PATH.exists():
    print(f"Skipping text extraction — set PDF_PATH to an existing file (got {PDF_PATH!r})")
else:
    job = client.extract_text(PDF_PATH, extractor="pymupdf", markdown=True)
    #job = client.extract_text(PDF_PATH, extractor="mineru", markdown=True)
    print(f"Enqueued: {job}")

    status = client.wait_for_job(
        job.job_id, 
        poll_interval=POLL_INTERVAL,
        timeout=JOB_TIMEOUT,
    )
    print(f"Finished: {status}")

    text_result = client.get_job_result(job.job_id)
    if isinstance(text_result, dict):
        preview = str(text_result)[:500]
    else:
        preview = str(text_result)[:500]
    print(f"Result preview:\n{preview}...")

Enqueued: JobResponse(job_id='fd4683fa-ca26-4738-afd3-9dc52c6108b7', status='queued', created_at='2026-11-25T12:51:19.865032', message='Text extraction job enqueued')


Finished: JobStatus(job_id='fd4683fa-ca26-4738-afd3-9dc52c6108b7', status='completed', created_at='2026-11-25T12:51:19.865032', started_at='2026-11-25T12:51:19.919415', completed_at='2026-11-25T12:52:45.540779', current_stage='text_extraction', completed_stages=['text_extraction'], error=None, progress=None)
Result preview:
{'text': "Revue des Études Grecques \n\n\n\nSur le troisième stasimon d'Œdipe-Roi Albert Machin \n\n#### Citer ce document / Cite this document : \n\nMachin Albert. Sur le troisième stasimon d'Œdipe-Roi. In: Revue des Études Grecques, tome 102, fascicule 485-486, Janvierjuin 1989. pp. 192-201 ; \n\ndoi : https://doi.org/10.3406/reg.1989.2445 \n\nhttps://www.persee.fr/doc/reg_0035-2039_1989_num_102_485_2445 \n\nFichier pdf généré le 18/04/2018 \n\n\n\n# SUR LE TROISIEME STASIMON O'ŒDIPE-ROI \n\n\...


In [5]:
print(text_result['text'][:500])

Revue des Études Grecques 



Sur le troisième stasimon d'Œdipe-Roi Albert Machin 

#### Citer ce document / Cite this document : 

Machin Albert. Sur le troisième stasimon d'Œdipe-Roi. In: Revue des Études Grecques, tome 102, fascicule 485-486, Janvierjuin 1989. pp. 192-201 ; 

doi : https://doi.org/10.3406/reg.1989.2445 

https://www.persee.fr/doc/reg_0035-2039_1989_num_102_485_2445 

Fichier pdf généré le 18/04/2018 



# SUR LE TROISIEME STASIMON O'ŒDIPE-ROI 



{O.R. vientLe de 1086-1 chant


## Extract references from markdown

Runs on the text extracted above. `method="full_text"` sends the whole text to the LLM; `method="semantic"` first locates the reference sections with embeddings.

In [6]:
#with open("../test_data/Holanda-Brasliadailyinvention-1989.md", "r") as f:
#    text_result = {}
#    text_result['text'] = f.read()

In [7]:
job = client.extract_references(
    text_result['text'],
    method="full_text",  # or "semantic"
    temperature=0.3,
)
print(f"Enqueued: {job}")

try:
    refs_result = client.wait_for_result(
        job.job_id,
        poll_interval=POLL_INTERVAL,
        timeout=JOB_TIMEOUT,
    )
    refs_result
except JobFailedError as e:
    print(f"Job failed: {e}")
except JobTimeoutError as e:
    print(f"Timed out after {e.timeout}s")

Enqueued: JobResponse(job_id='e55b83bd-6f7f-4380-91b9-1e3c75ef4e80', status='queued', created_at='2026-11-25T12:52:47.519057', message='Reference extraction job enqueued')


In [8]:
refs_result

{'references': ['C. M. Bowra, Sophoclean Tragedy, Oxford, 1944, p. 199.',
  'A. Lesky, Greek Tragedy, éd. angl. Londres, 1965, p. 113.',
  'R. C. Jebb éd. O.T. Cambridge, 1914, p. 143.',
  'J. C. Kamerbeek éd. O.T. Leyde, 1967, p. 208.',
  "R. W. B. Burton, The Chorus in Sophocles' Tragedies, Oxford, 1980, p. 169-70.",
  'H. D. F. Kitto, Greek Tragedy, 3e éd., Londres, 1970, p. 167.',
  'R. C. Jebb, op. cit., comm. à 1086-1109.',
  'W. Kranz, Stasimon, Berlin, 1933, p. 213.',
  'G. M. Kirkwood, A Study of Sophoclean Drama, New York, 1958, p. 199-200.',
  'J. C. Kamerbeek, op. cit., comm. à 1086-1109.',
  'T. B. L. Webster, An Introduction to Sophocles, Oxford, 1936, p. 105.',
  'Jebb; éd. Trachiniennes p. lxi, Antig. p. lxix, Ajax p. lxviii.',
  'M. A. Bayfield, éd. Antigone, N-Y, 1968, p. xxm.',
  'G. Mùller, éd. Antigone, Leyde, 1967, p. 242.',
  'J. C. Kamerbeek, éd. Trach. p. 142, Ant. p. 186, Aj. p. 146, O.T., p. 208.',
  'A. M. Dale, «Stasimon and Hyporchème», Eranos, 48, 1950, p

## Parse reference strings

Send raw bibliography lines to the parser endpoint.

In [9]:
#job = client.parse_references(refs_result['references'], parser="grobid", temperature=0.0)
#job = client.parse_references(refs_result['references'], parser="llm", temperature=0.0, prompt_name="prompts/reference_parsing.md")
#job = client.parse_references(refs_result['references'], parser="llm", temperature=0.0, prompt_name="prompts/reference_parsing_linkedbook.md")
job = client.parse_references(refs_result['references'], parser="llm", temperature=0.0)
print(f"Enqueued: {job}")

parsed_result = client.wait_for_result(
    job.job_id,
    poll_interval=POLL_INTERVAL,
    timeout=JOB_TIMEOUT,
)
parsed_result

Enqueued: JobResponse(job_id='ff7f9248-61e6-417e-9866-a10d3f4ffee9', status='queued', created_at='2026-11-25T12:53:43.678129', message='Reference parsing job enqueued')


{'references': [{'full_title': 'Sophoclean Tragedy',
   'container_title': None,
   'journal_title': None,
   'authors': [{'first_name': 'C.',
     'middle_name': 'M.',
     'surname': 'Bowra',
     'name_link': None,
     'role_name': None}],
   'editors': None,
   'publisher': 'Oxford',
   'translator': None,
   'publication_place': 'Oxford',
   'publication_year': 1944,
   'publication_date_raw': '1944',
   'identifiers': [],
   'ref_type': 'book',
   'raw': {},
   'volume': None,
   'issue': None,
   'pages': '199',
   'cited_range': '199',
   'footnote_number': None},
  {'full_title': 'Greek Tragedy',
   'container_title': None,
   'journal_title': None,
   'authors': [{'first_name': 'A.',
     'middle_name': None,
     'surname': 'Lesky',
     'name_link': None,
     'role_name': None}],
   'editors': None,
   'publisher': None,
   'translator': None,
   'publication_place': 'Londres',
   'publication_year': 1965,
   'publication_date_raw': '1965',
   'identifiers': [],
   'ref_t

In [10]:
len(parsed_result['references'])

65

## Link references to external indexes

`link_references` submits the **structured records** produced by the parser above
to `POST /link/references`, which resolves each one against OpenAlex, Matilda,
Wikidata and OpenCitations. Pass a list to submit a batch, or a single record.

Each target returns an `id` and a `doi`, both `None` when no confident match is
found. `method="model"` scores candidates with a model and sets `score`; a target
whose lookup failed sets `error` (the other targets still return). Linking hits
live external APIs once per reference per target, so keep `LINKING_LIMIT` small.

In [11]:
from citation_index_api_client import is_linkable

LINKING_TARGETS = "all"   # or "openalex", "matilda", "wikidata", "opencitations", or "openalex,wikidata"
LINKING_LIMIT = 5         # bounded: each reference hits every target's live API
LINKING_METHOD = "match"  # or "model" (adds a score per link)

# Linking consumes the parsed records, and the endpoint rejects the whole batch
# if any record lacks a title, authors or year — so keep only the linkable ones.
linkable = [r for r in parsed_result['references'] if is_linkable(r)]
print(f"{len(linkable)} of {len(parsed_result['references'])} records are linkable")
records_to_link = linkable[:LINKING_LIMIT]

job = client.link_references(records_to_link, target=LINKING_TARGETS, method=LINKING_METHOD)
print(f"Enqueued: {job}")

linked = client.wait_for_links(
    job.job_id,
    poll_interval=POLL_INTERVAL,
    timeout=JOB_TIMEOUT,
)
print(f"Linked {len(linked)} references")

28 of 65 records are linkable
Enqueued: JobResponse(job_id='cc489913-e82c-4382-a411-3d568860ddb0', status='queued', created_at='2026-11-25T12:53:48.020565', message='Citation linking job enqueued')


Linked 5 references


In [12]:
for item in linked:
    title = item.reference.get("full_title") or item.reference
    print(f"\n{str(title)[:90]}")
    for target, link in item.links.items():
        mark = "✓" if link.is_linked else "·"
        extra = f"  score={link.score:.3f}" if link.score is not None else ""
        extra += f"  error={link.error}" if link.error else ""
        print(f"  {mark} {target:<13} id={link.id or '—'}  doi={link.doi or '—'}{extra}")

hits = sum(link.is_linked for item in linked for link in item.links.values())
lookups = sum(len(item.links) for item in linked)
print(f"\n{hits} / {lookups} target lookups matched")


Sophoclean Tragedy
  · openalex      id=—  doi=—  error=OpenAlex daily budget exhausted; remaining=0; reset_seconds=39972
  · matilda       id=—  doi=—
  ✓ wikidata      id=Q123702058  doi=—
  · opencitations id=—  doi=—

Greek Tragedy
  · openalex      id=—  doi=—  error=OpenAlex daily budget exhausted; remaining=0; reset_seconds=39968
  · matilda       id=—  doi=—
  · wikidata      id=—  doi=—
  · opencitations id=—  doi=—

The Chorus in Sophocles' Tragedies
  · openalex      id=—  doi=—  error=OpenAlex daily budget exhausted; remaining=0; reset_seconds=39965
  · matilda       id=—  doi=—
  ✓ wikidata      id=Q123744614  doi=—
  · opencitations id=—  doi=—

Greek Tragedy
  · openalex      id=—  doi=—  error=OpenAlex daily budget exhausted; remaining=0; reset_seconds=39750
  · matilda       id=—  doi=—
  · wikidata      id=—  doi=—
  · opencitations id=—  doi=—

Stasimon
  · openalex      id=—  doi=—  error=OpenAlex daily budget exhausted; remaining=0; reset_seconds=39746
  · matilda

## GROBID pipeline: PDF → references

`process_references_grobid` goes straight from a PDF to index-aligned raw
`references` and `parsed_references` in one job — no separate text extraction or
parsing. With `include_context=True` it also returns `contexts`: the citing
sentences of each reference, ready for the classifier below.

In [13]:
job = client.process_references_grobid(PDF_PATH, include_context=True)
print(f"Enqueued: {job}")

grobid_result = client.wait_for_result(
    job.job_id,
    poll_interval=POLL_INTERVAL,
    timeout=JOB_TIMEOUT,
)
print(f"{len(grobid_result['references'])} references, "
      f"{len(grobid_result['parsed_references'])} parsed, "
      f"{len(grobid_result.get('contexts', []))} citing contexts")
grobid_result['contexts'][:3]

Enqueued: JobResponse(job_id='7a5b4fa2-2498-4903-874f-66fb94a5351a', status='queued', created_at='2026-11-25T12:57:44.747130', message='GROBID reference extraction job enqueued')


29 references, 29 parsed, 29 citing contexts


[[], [], []]

`/extract/references` also accepts a PDF upload, which uses GROBID.
`extract_references_from_pdf` can include parsed records and citing sentences too.

In [14]:
job = client.extract_references_from_pdf(PDF_PATH, include_parsed=True, include_context=False)
pdf_refs = client.wait_for_result(job.job_id, poll_interval=POLL_INTERVAL, timeout=JOB_TIMEOUT)
print(sorted(pdf_refs))
pdf_refs['references'][:5]

['count', 'method', 'parsed_references', 'references']


['C. M. Bowra, Sophoclean Tragedy, Oxford, 1944, p. 199.',
 'Par ex. A. Lesky, Greek Tragedy, éd. angl. Londres, 1965, p. 113.',
 'R. C. Jebb éd. O.T. Cambridge, 1914, p. 143. J. C. Kamerbeek éd. O.T. Leyde, 1967, p. 208.',
 "R. W. B. Burton, The Chorus in Sophocles' Tragedies, Oxford, 1980, p. 169-70.",
 'H. D. F. Kitto, Greek Tragedy, 3e éd., Londres, 1970, p. 167 : «lyrical instrument».']

## Classify citation intent

`classify_citations` labels each citing sentence as **method**, **background** or
**result**. Each item is `{"section": ..., "citation": ...}` (section may be `""`);
GROBID `contexts` (one list per reference) are flattened first. `mode` is `"WS"` (with sections), `"WoS"`
(without) or `"M"` (per item, default).

In [15]:
CLASSIFY_LIMIT = 10  # keep small while exploring

# contexts holds one list of citing sentences per reference; flatten it.
citing_sentences = [ctx for per_ref in grobid_result['contexts'] for ctx in per_ref][:CLASSIFY_LIMIT]
if not citing_sentences:
    print("No citing sentences found (footnote-only papers have none); skipping classification")
else:
    job = client.classify_citations(citing_sentences, mode="M")
    print(f"Enqueued: {job}")
    intents = client.wait_for_result(job.job_id, poll_interval=POLL_INTERVAL, timeout=JOB_TIMEOUT)
    print(intents)

No citing sentences found (footnote-only papers have none); skipping classification


## Citing paper metadata

`extract_metadata` returns the PDF's own record (title, authors, year, venue,
identifiers) as a `Reference`. `method` is `"grobid"` (default) or `"llm"`.
The job fails when no metadata is found.

In [16]:
job = client.extract_metadata(PDF_PATH, method="grobid")  # or "llm"
metadata = client.wait_for_result(job.job_id, poll_interval=POLL_INTERVAL, timeout=JOB_TIMEOUT)["metadata"]
metadata

{'full_title': None,
 'container_title': None,
 'journal_title': None,
 'authors': [{'first_name': 'Albert',
   'middle_name': None,
   'surname': 'Machin',
   'name_link': None,
   'role_name': None}],
 'editors': None,
 'publisher': None,
 'translator': None,
 'publication_place': None,
 'publication_year': None,
 'publication_date_raw': None,
 'identifiers': [{'scheme': 'doi',
   'value': '10.3406/reg.1989.2445',
   'normalized': '10.3406/reg.1989.2445'}],
 'ref_type': None,
 'raw': {},
 'volume': None,
 'issue': None,
 'pages': None,
 'cited_range': None,
 'footnote_number': None}

## OpenCitations deposit

`opencitations_deposit` runs the whole flow in one call: GROBID references,
citing-paper metadata, linking of the linkable records, then one OpenCitations
crowdsourcing deposit (META-CSV, the `===###===@@@===` line, CITS-CSV).
Options: `link=False`, `target=`, `link_method="model"`, `metadata_method="llm"`,
or `citing=CitingDocument(...)` to type the citing paper in. Markdown files need `citing=`.
Linking every reference takes a while.

In [17]:
deposit = client.opencitations_deposit(PDF_PATH, poll_interval=POLL_INTERVAL, timeout=JOB_TIMEOUT * 3)

with_ids = sum(not row["id"].startswith("temp:") for row in deposit.meta[1:])
print(f"citing: {deposit.meta[0]['id']}")
print(f"{len(deposit.meta) - 1} cited works ({with_ids} with real ids), "
      f"{len(deposit.cits)} citations, {len(deposit.dropped)} dropped, {deposit.merged} merged")
for d in deposit.dropped[:5]:
    print(f"  dropped #{d.record}: {d.reason}")

Path("deposit.csv").write_text(deposit.text, encoding="utf-8")
print(deposit.text[:800])

citing: doi:10.3406/reg.1989.2445
18 cited works (3 with real ids), 18 citations, 11 dropped, 0 merged
  dropped #3: short form or no title, and no identifier
  dropped #7: short form or no title, and no identifier
  dropped #9: no identifier, and a book row needs pub_date
  dropped #12: short form or no title, and no identifier
  dropped #13: short form or no title, and no identifier
"id","title","author","pub_date","venue","volume","issue","page","type","publisher","editor"
"doi:10.3406/reg.1989.2445","","Machin, Albert","","","","","","","",""
"wikidata:Q123702058","Sophoclean Tragedy","Bowra, C M","1944","","","","199-199","book","",""
"temp:2","Greek Tragedy","Par Ex, A; Lesky,","1965","","","","113-113","book","",""
"wikidata:Q123744614","The Chorus in Sophocles' Tragedies","Burton, R W B","1980","","","","169-170","book","",""
"temp:5","Greek Tragedy","Kitto, H D F","1970","","","","167-167","book","",""
"temp:6","A Study of Sophoclean Drama","Kranz, W; Stasimon, G M; Kirkwood,"

## Manual job polling (optional)

Use this pattern when you want to inspect status between polls instead of blocking on `wait_for_result`.

In [18]:
# Re-use the last job_id from the cell above, or paste one here:
job_id = job.job_id

status = client.get_job_status(job_id)
print(status)
print("terminal?", status.is_terminal)

if status.status == "completed":
    result = client.get_job_result(job_id)
    print(result)

JobStatus(job_id='a9164707-7887-48a9-aad6-a0f4dd61cb88', status='completed', created_at='2026-11-25T12:57:49.736912', started_at=None, completed_at='2026-11-25T12:57:49.820599', current_stage='metadata_extraction', completed_stages=['metadata_extraction'], error=None, progress=None)
terminal? True
{'metadata': {'full_title': None, 'container_title': None, 'journal_title': None, 'authors': [{'first_name': 'Albert', 'middle_name': None, 'surname': 'Machin', 'name_link': None, 'role_name': None}], 'editors': None, 'publisher': None, 'translator': None, 'publication_place': None, 'publication_year': None, 'publication_date_raw': None, 'identifiers': [{'scheme': 'doi', 'value': '10.3406/reg.1989.2445', 'normalized': '10.3406/reg.1989.2445'}], 'ref_type': None, 'raw': {}, 'volume': None, 'issue': None, 'pages': None, 'cited_range': None, 'footnote_number': None}, 'method': 'grobid'}


In [19]:
client.close()
print("Client closed.")

Client closed.
